# AU Deconstruction RAG in Colab

This notebook builds a Retrieval-Augmented Generation (RAG) system using:
- `95` JSONL files and `364` extracted chunks
- `ChromaDB` for vector search
- `OpenAI` for embeddings + answer generation
- `Gradio` for a simple chat interface

The notebook is written so others can follow each step and reproduce results.


## 1) Install libraries

Run this once per Colab runtime.


In [ ]:
!pip -q install chromadb openai gradio pandas


## 2) Connect data path and set constants

Set paths for the chunk files and local Chroma persistence.
If your repository is in Google Drive, this path should work after mounting Drive.


In [ ]:
from pathlib import Path

# Adjust if your repo folder name/path differs in Drive
REPO_ROOT = Path('/content/drive/MyDrive/ACTIVE/AU_deconstruction_domain/RAG/RAG_repo')
DATA_DIR = REPO_ROOT / 'data' / 'kept_chunks' / 'Chunks'
CHROMA_DIR = REPO_ROOT / 'chroma_db'
COLLECTION_NAME = 'au_deconstruction_chunks'

print('REPO_ROOT:', REPO_ROOT)
print('DATA_DIR exists:', DATA_DIR.exists())
print('CHROMA_DIR:', CHROMA_DIR)


## 3) (Optional) Mount Google Drive

Use this if your data is stored in Drive.
Skip if files are already local in Colab.


In [ ]:
from google.colab import drive

drive.mount('/content/drive')


## 4) Load API key from Colab secrets

Create a secret named `OPENAI_API_KEY` in Colab:
- Left sidebar -> Secrets -> Add new secret
- Name: `OPENAI_API_KEY`
- Value: your API key

This keeps the key out of code and notebooks.


In [ ]:
from google.colab import userdata

OPENAI_API_KEY = userdata.get('OPENAI_API_KEY')
if not OPENAI_API_KEY:
    raise ValueError('Missing OPENAI_API_KEY in Colab secrets.')

print('API key loaded from Colab secrets.')


## 5) Read all JSONL chunks

Each JSONL line is one chunk record.
We load records and keep key fields (`chunk_id`, `source_file`, `text`, etc.).


In [ ]:
import json
import pandas as pd

records = []
jsonl_files = sorted(DATA_DIR.glob('*.jsonl'))

for fp in jsonl_files:
    with fp.open('r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            obj = json.loads(line)
            text = (obj.get('text') or '').strip()
            if not text:
                continue
            records.append({
                'chunk_id': obj.get('chunk_id'),
                'source_file': obj.get('source_file'),
                'page_num': obj.get('page_num'),
                'chunk_index': obj.get('chunk_index'),
                'relevance': obj.get('relevance'),
                'text': text,
                'file_name': fp.name,
            })

chunks_df = pd.DataFrame(records)
print('JSONL files:', len(jsonl_files))
print('Chunks loaded:', len(chunks_df))
chunks_df.head(3)


## 6) Build ChromaDB from scratch

This section deletes the local Chroma folder and rebuilds the index every run.
That guarantees a clean and reproducible state.


In [ ]:
import shutil
import chromadb
from chromadb.utils.embedding_functions import OpenAIEmbeddingFunction

if CHROMA_DIR.exists():
    shutil.rmtree(CHROMA_DIR)
CHROMA_DIR.mkdir(parents=True, exist_ok=True)

client = chromadb.PersistentClient(path=str(CHROMA_DIR))

embedding_fn = OpenAIEmbeddingFunction(
    api_key=OPENAI_API_KEY,
    model_name='text-embedding-3-small'
)

collection = client.get_or_create_collection(
    name=COLLECTION_NAME,
    embedding_function=embedding_fn,
    metadata={'hnsw:space': 'cosine'}
)

ids = chunks_df['chunk_id'].fillna('').astype(str).tolist()
docs = chunks_df['text'].tolist()
metadatas = chunks_df[['source_file', 'page_num', 'chunk_index', 'relevance', 'file_name']].to_dict(orient='records')

safe_ids = []
seen = set()
for i, cid in enumerate(ids):
    cid = cid.strip() or f'chunk_{i}'
    if cid in seen:
        cid = f'{cid}__dup_{i}'
    seen.add(cid)
    safe_ids.append(cid)

BATCH = 100
for i in range(0, len(docs), BATCH):
    collection.add(
        ids=safe_ids[i:i+BATCH],
        documents=docs[i:i+BATCH],
        metadatas=metadatas[i:i+BATCH],
    )

print('Indexed chunks:', collection.count())


## 7) Test retrieval only

Before using the LLM, check that semantic retrieval returns relevant chunks.


In [ ]:
def retrieve(query, k=5):
    result = collection.query(query_texts=[query], n_results=k)
    return result

test_q = 'What policies support reuse and deconstruction in Australia?'
res = retrieve(test_q, k=3)

for i, (doc, meta, cid) in enumerate(zip(res['documents'][0], res['metadatas'][0], res['ids'][0]), start=1):
    print(f'--- Result {i} ---')
    print('chunk_id:', cid)
    print('source_file:', meta.get('source_file'))
    print('snippet:', doc[:400].replace('\n', ' '))
    print()


## 8) Create RAG answer function with citations

We retrieve top chunks, then ask OpenAI to answer only from that context.
The output includes source citations (`chunk_id`, file, page).


In [ ]:
from openai import OpenAI

llm_client = OpenAI(api_key=OPENAI_API_KEY)
LLM_MODEL = 'gpt-4.1-mini'

SYSTEM_PROMPT = (
    'You are a helpful assistant answering questions about Australian deconstruction, '
    'waste, and circular economy documents. Use only the provided context. '
    'If context is insufficient, say you do not have enough evidence.'
)

def build_context(query, k=6):
    result = collection.query(query_texts=[query], n_results=k)
    items = []
    for doc, meta, cid in zip(result['documents'][0], result['metadatas'][0], result['ids'][0]):
        items.append({
            'chunk_id': cid,
            'source_file': meta.get('source_file'),
            'page_num': meta.get('page_num'),
            'text': doc,
        })
    return items

def answer_with_rag(question, k=6):
    ctx_items = build_context(question, k=k)

    context_text = '\n\n'.join([
        f"[chunk_id: {it['chunk_id']}] [source_file: {it['source_file']}] [page_num: {it['page_num']}]\n{it['text']}"
        for it in ctx_items
    ])

    user_prompt = f"Question:\n{question}\n\nContext chunks:\n{context_text}\n\nInstructions:\n- Answer using only the context chunks.\n- Be concise and factual.\n- If insufficient evidence, explicitly say so.\n- End with a section titled 'Citations' listing chunk_id values used."

    resp = llm_client.responses.create(
        model=LLM_MODEL,
        input=[
            {'role': 'system', 'content': SYSTEM_PROMPT},
            {'role': 'user', 'content': user_prompt},
        ],
        temperature=0.1,
    )

    return resp.output_text, ctx_items

sample_answer, sample_ctx = answer_with_rag('What are practical reuse-related initiatives mentioned in the reports?')
print(sample_answer)


## 9) Launch simple Gradio chat app

This UI lets users ask questions and receive RAG answers with citations.


In [ ]:
import gradio as gr


def chat_fn(message, history):
    answer, ctx_items = answer_with_rag(message, k=6)

    citation_lines = ['\n\nRetrieved chunks:']
    for it in ctx_items:
        citation_lines.append(
            f"- {it['chunk_id']} | {it['source_file']} | page: {it['page_num']}"
        )

    return answer + '\n'.join(citation_lines)

examples = [
    'What does the dataset say about policy levers for deconstruction and material reuse?',
    'Summarize barriers to construction and demolition waste circularity in Australia.',
    'What evidence is there for take-back, repair, or reuse programs?',
]

ui = gr.ChatInterface(
    fn=chat_fn,
    title='AU Deconstruction RAG Demo',
    description='Ask questions. Answers are grounded in the indexed chunk dataset and include citations.',
    examples=examples,
)

ui.launch(share=True, debug=False)


## 10) What we built

- Loaded and validated chunk data (`95` files, `364` chunks).
- Built a Chroma vector index from scratch.
- Implemented retrieval + OpenAI answer generation.
- Exposed a simple Gradio chat app with citations.

You can now publish this repository with both data and runnable notebook.
